<a href="https://colab.research.google.com/github/DArturo-LR/Analizador-de-Datos-Demogr-ficos/blob/main/Analizador_de_datos_demograficos2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [9]:
pip install ucimlrepo

In [10]:
from ucimlrepo import fetch_ucirepo

# fetch dataset
adult = fetch_ucirepo(id=2)

# data (as pandas dataframes)
X = adult.data.features
y = adult.data.targets

# metadata
print(adult.metadata)

# variable information
print(adult.variables)


{'uci_id': 2, 'name': 'Adult', 'repository_url': 'https://archive.ics.uci.edu/dataset/2/adult', 'data_url': 'https://archive.ics.uci.edu/static/public/2/data.csv', 'abstract': 'Predict whether annual income of an individual exceeds $50K/yr based on census data. Also known as "Census Income" dataset. ', 'area': 'Social Science', 'tasks': ['Classification'], 'characteristics': ['Multivariate'], 'num_instances': 48842, 'num_features': 14, 'feature_types': ['Categorical', 'Integer'], 'demographics': ['Age', 'Income', 'Education Level', 'Other', 'Race', 'Sex'], 'target_col': ['income'], 'index_col': None, 'has_missing_values': 'yes', 'missing_values_symbol': 'NaN', 'year_of_dataset_creation': 1996, 'last_updated': 'Tue Sep 24 2024', 'dataset_doi': '10.24432/C5XW20', 'creators': ['Barry Becker', 'Ronny Kohavi'], 'intro_paper': None, 'additional_info': {'summary': "Extraction was done by Barry Becker from the 1994 Census database.  A set of reasonably clean records was extracted using the fol

In [11]:
import pandas as pd
from ucimlrepo import fetch_ucirepo

def calculate_demographic_data(print_data=True):
    # 1. Descargar el dataset usando la librería oficial
    adult = fetch_ucirepo(id=2)

    X = adult.data.features
    y = adult.data.targets

    # 2. Combinar X e y en un único DataFrame para poder cruzar los datos fácilmente
    df = pd.concat([X, y], axis=1)

    # NOTA: La librería ucimlrepo a veces conserva espacios iniciales en los textos.
    # Con esta línea limpiamos todas las columnas de texto para evitar fallos al filtrar.
    df = df.apply(lambda x: x.str.strip() if x.dtype == "object" else x)

    # ------------------ EMPIEZAN LOS CÁLCULOS ------------------

    # 1. ¿Cuántas personas de cada raza están representadas?
    race_count = df['race'].value_counts()

    # 2. ¿Cuál es la edad promedio de los hombres?
    average_age_men = round(df[df['sex'] == 'Male']['age'].mean(), 1)

    # 3. ¿Cuál es el porcentaje de personas que tienen un grado de licenciatura?
    percentage_bachelors = round((df['education'] == 'Bachelors').sum() / len(df) * 100, 1)

    # 4. Educación avanzada (>50K o >50K.)
    # Nota: A veces en UCI el target viene con un punto final ('>50K.'). Usamos .isin() para prevenirlo.
    higher_education = df['education'].isin(['Bachelors', 'Masters', 'Doctorate'])
    lower_education = ~higher_education

    # Porcentaje con educación avanzada que gana más de 50k
    df_higher = df[higher_education]
    higher_education_rich = round((df_higher['income'].isin(['>50K', '>50K.'])).sum() / len(df_higher) * 100, 1)

    # 5. Sin educación avanzada que gana más de 50k
    df_lower = df[lower_education]
    lower_education_rich = round((df_lower['income'].isin(['>50K', '>50K.'])).sum() / len(df_lower) * 100, 1)

    # 6. Mínimo de horas semanales
    min_work_hours = df['hours-per-week'].min()

    # 7. Porcentaje de ricos que trabajan el mínimo de horas
    num_min_workers = df[df['hours-per-week'] == min_work_hours]
    rich_percentage = round((num_min_workers['income'].isin(['>50K', '>50K.'])).sum() / len(num_min_workers) * 100, 1)

    # 8. País con mayor porcentaje de altos ingresos
    country_counts = df['native-country'].value_counts()
    country_rich_counts = df[df['income'].isin(['>50K', '>50K.'])]['native-country'].value_counts()

    country_rich_percentage = (country_rich_counts / country_counts) * 100

    highest_earning_country = country_rich_percentage.idxmax()
    highest_earning_country_percentage = round(country_rich_percentage.max(), 1)

    # 9. Ocupación más popular para >50K en India
    india_rich = df[(df['native-country'] == 'India') & (df['income'].isin(['>50K', '>50K.']))]
    top_IN_occupation = india_rich['occupation'].value_counts().idxmax()

    # ------------------ IMPRESIÓN DE RESULTADOS ------------------
    if print_data:
        print("Number of each race:\n", race_count)
        print("Average age of men:", average_age_men)
        print(f"Percentage with Bachelors degrees: {percentage_bachelors}%")
        print(f"Percentage with higher education that earn >50K: {higher_education_rich}%")
        print(f"Percentage without higher education that earn >50K: {lower_education_rich}%")
        print(f"Min work time: {min_work_hours} hours/week")
        print(f"Percentage of rich among those who work fewest hours: {rich_percentage}%")
        print("Highest earning country:", highest_earning_country)
        print(f"Highest earning country percentage: {highest_earning_country_percentage}%")
        print("Top occupations in India:", top_IN_occupation)

    return {
        'race_count': race_count,
        'average_age_men': average_age_men,
        'percentage_bachelors': percentage_bachelors,
        'higher_education_rich': higher_education_rich,
        'lower_education_rich': lower_education_rich,
        'min_work_hours': min_work_hours,
        'rich_percentage': rich_percentage,
        'highest_earning_country': highest_earning_country,
        'highest_earning_country_percentage': highest_earning_country_percentage,
        'top_IN_occupation': top_IN_occupation
    }

In [12]:
# EJECUTA ESTA CELDA PARA VER LOS RESULTADOS
print("--- EJECUTANDO ANÁLISIS DEMOGRÁFICO ---")
resultados = calculate_demographic_data()

--- EJECUTANDO ANÁLISIS DEMOGRÁFICO ---
Number of each race:
 race
White                 41762
Black                  4685
Asian-Pac-Islander     1519
Amer-Indian-Eskimo      470
Other                   406
Name: count, dtype: int64
Average age of men: 39.5
Percentage with Bachelors degrees: 16.4%
Percentage with higher education that earn >50K: 46.1%
Percentage without higher education that earn >50K: 17.3%
Min work time: 1 hours/week
Percentage of rich among those who work fewest hours: 11.1%
Highest earning country: France
Highest earning country percentage: 42.1%
Top occupations in India: Prof-specialty
